In [3]:
import Pkg; Pkg.activate(joinpath(@__DIR__, "..")); Pkg.instantiate();
using RobotZoo: YakPlane
import RobotDynamics
using ForwardDiff
using StaticArrays
using LinearAlgebra
using Rotations
using Printf
using Test
using TrajOptPlots
using JLD2
const RD = RobotDynamics

  Activating new project at `~/Documents/Robotics PhD/Fourth Semester/15-780 Grad AI`


RobotDynamics

In [9]:
g = 9.81; #Gravitational acceleration (m/s^2)
rho = 1.2; #Air density at 20C (kg/m^3)
m = .075; #Mass of plane (kg)

Jx = 4.8944e-04; #roll axis inertia (kg*m^2)
Jy = 6.3778e-04; #pitch axis inertia (kg*m^2)
Jz = 7.9509e-04; #yaw axis inertia (kg*m^2)

J = Diagonal(@SVector [Jx,Jy,Jz]);
Jinv= Diagonal(@SVector [1/Jx,1/Jy,1/Jz]); #Assuming products of inertia are small

Jm = .007*(.0075)^2 + .002*(.14)^2/12; #motor + prop inertia (kg*m^2)

# All lifting surfaces are modeled as unsweapt tapered wings
b = 45/100; #wing span (m)
l_in = 6/100; #inboard wing length covered by propwash (m)
cr = 13.5/100; #root chord (m)
ct = 8/100; #tip chord (m)
cm = (ct + cr)/2; #mean wing chord (m)
S = b*cm; #planform area of wing (m^2)
S_in = 2*l_in*cr;
S_out = S-S_in;
#Ra::T = b^2/S; %wing aspect ratio (dimensionless)
Rt = ct/cr; #wing taper ratio (dimensionless)
r_ail = (b/6)*(1+2*Rt)/(1+Rt); #aileron moment arm (m)

ep_ail = 0.63; #flap effectiveness (Phillips P.41)
trim_ail = 106; #control input for zero deflection
g_ail = (15*pi/180)/100; #maps control input to deflection angle

b_elev = 16/100; #elevator span (m)
cr_elev = 6/100; #elevator root chord (m)
ct_elev = 4/100; #elevator tip chord (m)
cm_elev = (ct_elev + cr_elev)/2; #mean elevator chord (m)
S_elev = b_elev*cm_elev; #planform area of elevator (m^2)
Ra_elev = b_elev^2/S_elev; #wing aspect ratio (dimensionless)
r_elev = 22/100; #elevator moment arm (m)

ep_elev = 0.88; #flap effectiveness (Phillips P.41)
trim_elev = 106; #control input for zero deflection
g_elev = (20*pi/180)/100; #maps control input to deflection angle

b_rud = 10.5/100; #rudder span (m)
cr_rud = 7/100; #rudder root chord (m)
ct_rud = 3.5/100; #rudder tip chord (m)
cm_rud = (ct_rud + cr_rud)/2; #mean rudder chord (m)
S_rud = b_rud*cm_rud; #planform area of rudder (m^2)
Ra_rud = b_rud^2/S_rud; #wing aspect ratio (dimensionless)
r_rud = 24/100; #rudder moment arm (m)
z_rud = 2/100; #height of rudder center of pressure (m)

ep_rud = 0.76; #flap effectiveness (Phillips P.41)
trim_rud = 106; #control input for zero deflection
g_rud = (35*pi/180)/100; #maps from control input to deflection angle

trim_thr = 24; #control input for zero thrust (deadband)
g_thr = 0.006763; #maps control input to Newtons of thrust
g_mot = 3000*2*pi/60*7/255; #maps control inp

In [18]:
#Some standard functions for dealing with rotation matrices and quaternions from the class notes

function hat(ω)
    return [0    -ω[3]  ω[2];
            ω[3]  0    -ω[1];
           -ω[2]  ω[1]  0   ]
end

function L(q)
    [q[1] -q[2:4]'; q[2:4] q[1]*I + hat(q[2:4])]
end

function R(q)
    [q[1] -q[2:4]'; q[2:4] q[1]*I - hat(q[2:4])]
end

const H = [zeros(1,3); I];

In [48]:

function yak_dynamics(x::StaticVector, u::StaticVector, t=0)
    r,q,v,w = x[1:3],x[4:7],x[8:10],x[11:13]

    Q = H'*L(q)*R(q)'*H
    #display(Q)

    # control input
    thr  = u[1]; #Throttle command (0-255 as sent to RC controller)
    ail  = u[2]; #Aileron command (0-255 as sent to RC controller)
    elev = u[3]; #Elevator command (0-255 as sent to RC controller)
    rud  = u[4]; #Rudder command (0-255 as sent to RC controller)

    #Note that body coordinate frame is:
    # x: points forward out nose
    # y: points out right wing tip
    # z: points down

    # ------- Input Checks -------- #
    thr  = clamp(thr,  0, 255)
    ail  = clamp(ail,  0, 255)
    elev = clamp(elev, 0, 255)
    rud  = clamp(rud,  0, 255)

    # ---------- Map Control Inputs to Angles ---------- #
    delta_ail = (ail-trim_ail)*g_ail;
    delta_elev = (elev-trim_elev)*g_elev;
    delta_rud = (rud-trim_rud)*g_rud;

    # ---------- Aerodynamic Forces (body frame) ---------- #
    v_body = Q'*v; #body-frame velocity
    v_rout = v_body + cross(w, @SVector [0,  r_ail, 0]);
    v_lout = v_body + cross(w, @SVector [0, -r_ail, 0]);
    v_rin  = v_body + cross(w, @SVector [0,  l_in, 0]) + propwash(thr);
    v_lin  = v_body + cross(w, @SVector [0, -l_in, 0]) + propwash(thr);
    v_elev = v_body + cross(w, @SVector [-r_elev, 0, 0]) + propwash(thr);
    v_rud  = v_body + cross(w, @SVector [-r_rud,  0, -z_rud]) + propwash(thr);

    # --- Outboard Wing Sections --- #
    a_rout = alpha(v_rout);
    a_lout = alpha(v_lout);
    a_eff_rout = a_rout + ep_ail*delta_ail; #effective angle of attack
    a_eff_lout = a_lout - ep_ail*delta_ail; #effective angle of attack

    F_rout = -p_dyn(v_rout)*.5*S_out* @SVector [Cd_wing(a_eff_rout), 0, Cl_wing(a_eff_rout)];
    F_lout = -p_dyn(v_lout)*.5*S_out* @SVector [Cd_wing(a_eff_lout), 0, Cl_wing(a_eff_lout)];

    F_rout = arotate(a_rout,F_rout); #rotate to body frame
    F_lout = arotate(a_lout,F_lout); #rotate to body frame

    # --- Inboard Wing Sections (Includes Propwash) --- #
    a_rin = alpha(v_rin);
    a_lin = alpha(v_lin);
    a_eff_rin = a_rin + ep_ail*delta_ail; #effective angle of attack
    a_eff_lin = a_lin - ep_ail*delta_ail; #effective angle of attack

    F_rin = -p_dyn( v_rin)*.5*S_in* @SVector [Cd_wing(a_eff_rin), 0, Cl_wing(a_eff_rin)];
    F_lin = -p_dyn( v_lin)*.5*S_in* @SVector [Cd_wing(a_eff_lin), 0, Cl_wing(a_eff_lin)];

    F_rin = arotate(a_rin,F_rin); #rotate to body frame
    F_lin = arotate(a_lin,F_lin); #rotate to body frame

    # println("AoA: right: $(rad2deg(a_rout)), left: $(rad2deg(a_lout))")

    # --- Elevator --- #
    a_elev = alpha(v_elev);
    a_eff_elev = a_elev + ep_elev*delta_elev; #effective angle of attack

    F_elev = -p_dyn(v_elev)*S_elev*@SVector [Cd_elev(a_eff_elev), 0, Cl_plate(a_eff_elev)];

    F_elev = arotate(a_elev,F_elev); #rotate to body frame

    # --- Rudder --- #
    a_rud = beta(v_rud);
    a_eff_rud = a_rud - ep_rud*delta_rud; #effective angle of attack

    F_rud = -p_dyn(v_rud)*S_rud*@SVector [Cd_rud(a_eff_rud), Cl_plate(a_eff_rud), 0];

    F_rud = brotate(a_rud,F_rud); #rotate to body frame

    # --- Thrust --- #
    if thr > trim_thr
        F_thr = @SVector [(thr-trim_thr)*g_thr, 0, 0];
        w_mot = @SVector [g_mot*thr, 0, 0];
    else #deadband
        F_thr = @SVector zeros(3);
        w_mot = @SVector zeros(3);
    end

    # ---------- Aerodynamic Torques (body frame) ---------- #
    T_rout = cross((@SVector [0, r_ail, 0]),F_rout);
    T_lout = cross((@SVector [0, -r_ail, 0]),F_lout);

    T_rin = cross((@SVector [0, l_in, 0]),F_rin);
    T_lin = cross((@SVector [0, -l_in, 0]),F_lin);

    T_elev = cross((@SVector [-r_elev, 0, 0]),F_elev);

    T_rud = cross((@SVector [-r_rud, 0, -z_rud]),F_rud);

    # ---------- Add Everything Together ---------- #
    # problems: F_lout, F_rin
    F_aero = F_rout + F_lout + F_rin + F_lin + F_elev + F_rud + F_thr;
    F = Q*F_aero - @SVector [0, 0, m*g];
    display(F)

    T = T_rout + T_lout + T_rin + T_lin + T_elev + T_rud + cross((J*w + Jm*w_mot),w);

    rdot = v;
    qdot = 0.5*L(q)*H*w
    vdot = F/m
    wdot = Jinv*T

    # xdot = [v;
    #         .25*((1-r'*r)*w - 2*cross(w,r) + 2*(w'*r)*r);
    #         F/p.m;
    #         p.Jinv*T];
    #
    #RobotDynamics.build_state(p, rdot, qdot, vdot, wdot)
    return[rdot;qdot;vdot;wdot]
end

@generated function dynamics!(xdot, x, u) where R
    if R <: UnitQuaternion
        Nx = 14
    else
        Nx = 12
    end
    quote
        xstatic = SVector{$Nx}(x)
        ustatic = SVector{4}(u)
        xdot .= dynamics(model, x, u)
        return nothing
    end
end

"Angle of attack"
@inline alpha(v) = atan(v[3],v[1])

"Sideslip angle"
@inline beta(v) = atan(v[2],v[1])

"Rotate by angle of attack"
function arotate(a,r)
    sa,ca = sincos(a)
    R = @SMatrix [
        ca 0 -sa;
        0  1   0;
        sa 0  ca]
    R*r
end

"Rotate by sideslip angle"
function brotate(b,r)
    sb,cb = sincos(b)
    R = @SMatrix [
        cb -sb 0;
        sb  cb 0;
        0    0 1]
    R*r
end

""" Propwash wind speed (body frame)
Fit from anemometer data taken at tail
No significant different between wing/tail measurements
"""
function propwash(thr)::SVector{3}
    trim_thr = 24; # control input for zero thrust (deadband)
    if thr > trim_thr
        v = @SVector [5.568*thr^0.199 - 8.859, 0, 0];
    else #deadband
        v = @SVector zeros(3)
    end
end

"Dynamic pressure"
function p_dyn( v)
    pd = 0.5*rho*(v'v)
end

""" Lift coefficient (alpha in radians)
3rd order polynomial fit to glide-test data
Good to about ±20°
"""
function Cl_wing(a)
    a = clamp(a, -0.5*pi, 0.5*pi)
    cl = -27.52*a^3 - .6353*a^2 + 6.089*a;
end

""" Lift coefficient (alpha in radians)
Ideal flat plate model used for wing and rudder
"""
function Cl_plate(a)
    a = clamp(a, -0.5*pi, 0.5*pi)
    cl = 2pi*a
end

""" Drag coefficient (alpha in radians)
2nd order polynomial fit to glide-test data
Good to about ±20°
"""
function Cd_wing(a)
    a = clamp(a, -0.5*pi, 0.5*pi)
    cd = 2.08*a^2 + .0612;
end

""" Drag coefficient (alpha in radians)
Induced drag for a tapered finite wing
    From phillips P.55
"""
function Cd_elev(a)
    a = clamp(a, -0.5*pi, 0.5*pi)
    cd = (4*pi*a^2)/Ra_elev
end

""" Drag coefficient (alpha in radians)
Induced drag for a tapered finite wing
From Phillips P.55
"""
function Cd_rud(a)
    a = clamp(a, -0.5*pi, 0.5*pi)
    cd = (4*pi*a^2)/Ra_rud;
end

Cd_rud

In [49]:
x = @SVector[-3.0;0.0;1.5;-6.114599630220157e-8;0.997155939027903;0.0;0.07536599539167085;5.0;0.0;0.0;0.0;0.0;0.0]
u = @SVector[141.5706794868393;133.69414194643682;34.33023247673691;110.62496144064313]
a = yak_dynamics(x, u)

3-element SVector{3, Float64} with indices SOneTo(3):
  0.5969895559801405
 -0.054276888070457156
 -0.34600602880379844

13-element Vector{Float64}:
   5.0
   0.0
   0.0
   0.0
   0.0
   0.0
   0.0
   7.959860746401874
  -0.7236918409394287
  -4.613413717383979
 -53.22641165343696
 190.64164585614506
 -16.7024753184936